In [7]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader
from torch.optim import AdamW


In [9]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Loading a public sentiment dataset to serve as our initial test base
print("Loading dataset")
raw_dataset = load_dataset("imdb")

Loading weights: 100%|██████████████| 197/197 [00:00<00:00, 6492.78it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading dataset


In [10]:
def tokenize_function(examples):
	# Truncating to 128 to speed up our initial testing phase
	return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data")
tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)

# Renaming the column so the model recognizes it as the targets
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Setting the format to PyTorch tensors
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized_datasets["train"].select(range(2000))
dataset_size = len(train_dataset)
print(dataset_size)

Tokenizing data


2000


In [11]:
# Filtering the dataset by class
class_0_dataset = tokenized_datasets["train"].filter(lambda example: example["labels"] == 0)
class_1_dataset = tokenized_datasets["train"].filter(lambda example: example["labels"] == 1)

# Creating a 10:1 imbalance ratio
majority_dataset = class_0_dataset.shuffle(seed=42).select(range(int(dataset_size * 0.9)))
minority_dataset = class_1_dataset.shuffle(seed=42).select(range(int(dataset_size * 0.1)))

# Combine and shuffle final dataset
imbalanced_train_dataset = concatenate_datasets([majority_dataset, minority_dataset])
train_dataset = imbalanced_train_dataset.shuffle(seed=42)

In [12]:
# Setting batch size based on typical RoBERTa fine-tuning
batch_size = 16

# Creating the DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    shuffle=True, batch_size=batch_size
)

# Instantiating the optimizer with a standard learning rate for fine-tuning
optimizer = AdamW(model.parameters(), lr=5e-5)

In [13]:

# Number of times the model will see the entire dataset
num_epocs = 3
device = torch.device("cuda")
model.to(device)

# Setting the model to training mode
model.train()
for epoch in range(num_epocs):
    for step, batch in enumerate(train_dataloader):
        # Moving the data to the same device as the model
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Clear previus gradients
        optimizer.zero_grad()
        
        # Forward pass and loss extraction
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backpropagation
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        print(f"Batch {step + 1} | Loss {loss.item()}")


weights_dir = "../models/roberta_baseline"
os.makedirs(weights_dir, exist_ok=True)

# Save the model and the tokenizer
print("Saving model to", weights_dir)
model.save_pretrained(weights_dir)
tokenizer.save_pretrained(weights_dir)

Batch 1 | Loss 0.8533103466033936
Batch 2 | Loss 0.7083495855331421
Batch 3 | Loss 0.6374363303184509
Batch 4 | Loss 0.5190040469169617
Batch 5 | Loss 0.42541152238845825
Batch 6 | Loss 0.30438128113746643
Batch 7 | Loss 0.05495959892868996
Batch 8 | Loss 0.5135332345962524
Batch 9 | Loss 0.05716073140501976
Batch 10 | Loss 0.8609793782234192
Batch 11 | Loss 0.6378812789916992
Batch 12 | Loss 0.3728357255458832
Batch 13 | Loss 0.3450811803340912
Batch 14 | Loss 0.3959946036338806
Batch 15 | Loss 0.13200151920318604
Batch 16 | Loss 0.18034648895263672
Batch 17 | Loss 0.23542767763137817
Batch 18 | Loss 0.03479276970028877
Batch 19 | Loss 0.2578072249889374
Batch 20 | Loss 0.24547439813613892
Batch 21 | Loss 0.6632487773895264
Batch 22 | Loss 0.5260222554206848
Batch 23 | Loss 0.378428190946579
Batch 24 | Loss 0.5583586692810059
Batch 25 | Loss 0.3574982285499573
Batch 26 | Loss 0.16752810776233673
Batch 27 | Loss 0.26835235953330994
Batch 28 | Loss 0.21925701200962067
Batch 29 | Loss 0.

Writing model shards: 100%|███████████████| 1/1 [00:04<00:00,  4.89s/it]


('../models/roberta_baseline/tokenizer_config.json',
 '../models/roberta_baseline/tokenizer.json')